In [1]:
import pandas as pd 
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Embedding, LSTM, Dense

from tensorflow.keras.utils import to_categorical

from tensorflow.keras.callbacks import EarlyStopping

from sklearn.utils import shuffle

In [2]:
text = pd.read_csv("Dataset.csv")

In [3]:
text.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2467 entries, 0 to 2466
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   transcript  2467 non-null   object
 1   url         2467 non-null   object
dtypes: object(2)
memory usage: 38.7+ KB


In [4]:
text = text["transcript"]

In [5]:
text = text.astype(str).str.lower()

In [6]:
text

0       good morning. how are you?(laughter)it's been ...
1       thank you so much, chris. and it's truly a gre...
2       (music: "the sound of silence," simon & garfun...
3       if you're here today — and i'm very happy that...
4       about 10 years ago, i took on the task to teac...
                              ...                        
2462    so, ma was trying to explain something to me a...
2463    this is a picture of a sunset on mars taken by...
2464    in my early days as a graduate student, i went...
2465    i took a cell phone and accidentally made myse...
2466    we humans are becoming an urban species, so ci...
Name: transcript, Length: 2467, dtype: object

In [7]:
print(text.shape)

(2467,)


In [8]:
print(len(text))

2467


In [9]:
text = text[:200]

In [10]:
text

0      good morning. how are you?(laughter)it's been ...
1      thank you so much, chris. and it's truly a gre...
2      (music: "the sound of silence," simon & garfun...
3      if you're here today — and i'm very happy that...
4      about 10 years ago, i took on the task to teac...
                             ...                        
195    a great way to start, i think, with my view of...
196    you know, i've talked about some of these proj...
197    in this rather long sort of marathon presentat...
198    i grew up to study the brain because i have a ...
199    i'm going to go right into the slides. and all...
Name: transcript, Length: 200, dtype: object

In [11]:
tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n')
tokenizer.fit_on_texts(text)

In [12]:
word_index = tokenizer.word_index

total_words = len(word_index) + 1

print(word_index)

print("Vocabulary Size:", total_words)


{'the': 1, 'and': 2, 'to': 3, 'of': 4, 'a': 5, 'that': 6, 'in': 7, 'i': 8, 'is': 9, 'you': 10, 'it': 11, 'this': 12, 'we': 13, '—': 14, 'so': 15, 'was': 16, 'for': 17, 'on': 18, 'have': 19, 'are': 20, 'they': 21, 'but': 22, "it's": 23, 'what': 24, 'with': 25, 'about': 26, 'be': 27, 'can': 28, 'all': 29, 'do': 30, 'as': 31, 'at': 32, 'not': 33, 'one': 34, 'people': 35, 'if': 36, 'like': 37, 'my': 38, 'there': 39, 'know': 40, 'from': 41, 'just': 42, 'an': 43, 'because': 44, 'very': 45, 'out': 46, 'going': 47, 'these': 48, 'think': 49, 'or': 50, 'now': 51, 'laughter': 52, 'by': 53, "that's": 54, 'when': 55, 'up': 56, 'them': 57, 'he': 58, 'me': 59, 'which': 60, 'how': 61, 'our': 62, 'really': 63, 'get': 64, 'had': 65, "i'm": 66, 'see': 67, 'would': 68, 'more': 69, 'then': 70, 'here': 71, "don't": 72, 'things': 73, 'who': 74, 'world': 75, 'your': 76, 'were': 77, 'time': 78, 'some': 79, 'their': 80, 'want': 81, 'way': 82, 'years': 83, 'will': 84, 'go': 85, 'has': 86, 'into': 87, 'said': 88,

In [13]:
input_sequences = []

for line in text:

    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])

In [14]:
for sequence in input_sequences[:10]:
    print(sequence)

[120, 537]
[120, 537, 61]
[120, 537, 61, 20]
[120, 537, 61, 20, 10]
[120, 537, 61, 20, 10, 52]
[120, 537, 61, 20, 10, 52, 23]
[120, 537, 61, 20, 10, 52, 23, 100]
[120, 537, 61, 20, 10, 52, 23, 100, 173]
[120, 537, 61, 20, 10, 52, 23, 100, 173, 1967]
[120, 537, 61, 20, 10, 52, 23, 100, 173, 1967, 11]


In [15]:
max_sequence_len = max(len(seq) for seq in input_sequences)

In [16]:
max_sequence_len

5892

In [32]:
max_sequence_len = 10   # or 50, 200 depending on your data

input_sequences = pad_sequences(
    input_sequences,
    maxlen=max_sequence_len,
    padding="pre",
    truncating="pre"
)

In [33]:
print(input_sequences[:5])

[[  0   0   0   0   0   0   0   0 120 537]
 [  0   0   0   0   0   0   0 120 537  61]
 [  0   0   0   0   0   0 120 537  61  20]
 [  0   0   0   0   0 120 537  61  20  10]
 [  0   0   0   0 120 537  61  20  10  52]]


In [34]:
X = input_sequences[:, :-1]

y = input_sequences[:, -1]

In [35]:
print("X Shape:", X.shape)

print("y Shape:", y.shape)

X Shape: (521916, 9)
y Shape: (521916,)


In [36]:
print("Vocabulary Size:", total_words)
print("Maximum Sequence Length:", max_sequence_len)
print("Number of Training Sequences:", len(input_sequences))

Vocabulary Size: 21805
Maximum Sequence Length: 10
Number of Training Sequences: 521916


In [37]:
import warnings
warnings.filterwarnings("ignore")

In [38]:
model = Sequential([
    
    Embedding(input_dim=total_words,
              output_dim=100,
              input_length=max_sequence_len-1),

    LSTM(150),

    Dense(total_words, activation='softmax')

])


In [39]:
print(X.shape)
print(y.shape)


(521916, 9)
(521916,)


In [40]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [41]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [42]:
history = model.fit(
    X,
    y,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/5
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 674s 91ms/step - accuracy: 0.1027 - loss: 6.2351 - val_accuracy: 0.1384 - val_loss: 5.9770
Epoch 2/5
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 637s 87ms/step - accuracy: 0.1482 - loss: 5.5377 - val_accuracy: 0.1519 - val_loss: 5.8839
Epoch 3/5
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 621s 85ms/step - accuracy: 0.1658 - loss: 5.2067 - val_accuracy: 0.1566 - val_loss: 5.8988
Epoch 4/5
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 692s 94ms/step - accuracy: 0.1798 - loss: 4.9376 - val_accuracy: 0.1555 - val_loss: 5.9649
Epoch 5/5
7340/7340 ━━━━━━━━━━━━━━━━━━━━ 603s 82ms/step - accuracy: 0.1936 - loss: 4.6989 - val_accuracy: 0.1538 - val_loss: 6.0591


In [43]:
seed_text = "i'm going to go right into the slides."

next_words = 6

for _ in range(next_words):

    token_list = tokenizer.texts_to_sequences([seed_text])[0]

    token_list = pad_sequences(
        [token_list],
        maxlen=max_sequence_len-1,
        padding="pre"
    )

    predicted = model.predict(token_list, verbose=0)

    predicted_word = tokenizer.index_word[
        np.argmax(predicted)
    ]

    seed_text += " " + predicted_word

print(seed_text)

i'm going to go right into the slides. and i was a little bit
